# Experimento Colaborativo #01 · Catastrophe Analysis · Grupo B
**Karen Micaela García Ferrari** · Modalidad Gerencial · basado en `z611_WorkFlow_01_gerencial_grupoB` de la cátedra

**Pregunta:** ¿cuál es el mejor método para corregir las variables que en 202006 quedaron en cero por una falla del datawarehouse?

**Métodos que se comparan** (todo lo demás del workflow queda igual, *ceteris paribus*):
* `asignar_NA`: lo que hace la cátedra, pone NA en las variables rotas de 202006
* `dejar_ceros`: no corregir nada
* `interpolar`: reemplazar por el promedio del mismo cliente en 202005 y 202007

**Cómo se mide:** futuro = 202107 (clase conocida), ganancia suavizada máxima en millones, con 8 semillas por método.

## Cómo correrlo (3 corridas en total)
1. Runtime **Python 3** → correr las 2 primeras celdas de código (Drive y descarga).
2. Runtime **R** → en la celda *PARAMETROS* elegir el método y correr todo desde ahí.
   * Corrida 1: `metodo <- "asignar_NA"` y `gridsearch <- TRUE` (≈ 2 h: busca hiperparámetros y los guarda)
   * Corrida 2: `metodo <- "dejar_ceros"` y `gridsearch <- FALSE` (≈ 1 h)
   * Corrida 3: `metodo <- "interpolar"` y `gridsearch <- FALSE` (≈ 1 h)
3. Al final, correr la última celda (**Análisis**): tabla de resultados, test de Wilcoxon y gráfico.

Los resultados se acumulan en `labo1/exp/resultados_CA.txt` (en tu Google Drive), así que podés cerrar Colab entre corrida y corrida.

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"

## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()

# ================= LO UNICO QUE SE CAMBIA ENTRE CORRIDAS =================
PARAM$CA$metodo <- "asignar_NA"   # "asignar_NA" | "dejar_ceros" | "interpolar"
PARAM$gridsearch <- TRUE          # TRUE solo en la corrida 1 (asignar_NA), FALSE en las otras dos
# =========================================================================

# mis semillas (las mismas para los 3 metodos: comparacion pareada)
PARAM$semillas <- c(151729, 235877, 526777, 582037, 803273, 208057, 453029, 352493)
PARAM$semilla_primigenia <- PARAM$semillas[1]

PARAM$experimento <- paste0("CA_", PARAM$CA$metodo)
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# archivos compartidos entre las 3 corridas (quedan en Google Drive)
PARAM$archivo_resultados <- "/content/buckets/b1/exp/resultados_CA.txt"
PARAM$archivo_hiper <- "/content/buckets/b1/exp/CA_mejores_hiperparametros.txt"


#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

Se comparan tres métodos de corrección. El método se elige en la celda **PARAMETROS**.

In [ ]:
# Variables que en 202006 quedaron en cero por la falla del datawarehouse
# (la misma lista que usa la catedra)
campos_rotos <- c("internet", "mrentabilidad", "mrentabilidad_annual", "mcomisiones",
  "mactivos_margen", "mpasivos_margen", "mcuentas_saldo",
  "ctarjeta_visa_transacciones", "mtarjeta_visa_consumo", "mtarjeta_master_consumo",
  "ccallcenter_transacciones", "chomebanking_transacciones")
campos_rotos <- intersect(campos_rotos, colnames(dataset))
cat("Metodo:", PARAM$CA$metodo, " | campos tratados:", length(campos_rotos), "\n")

if (PARAM$CA$metodo == "asignar_NA") {
  # metodo de la catedra: la "anti imputacion"
  for (campo in campos_rotos) dataset[foto_mes == 202006, (campo) := NA]

} else if (PARAM$CA$metodo == "dejar_ceros") {
  # no se corrige nada: los ceros falsos quedan como estan

} else if (PARAM$CA$metodo == "interpolar") {
  # promedio del mismo cliente entre el mes anterior (202005) y el siguiente (202007)
  # si el cliente no tiene alguno de esos dos meses, queda NA
  setorder(dataset, numero_de_cliente, foto_mes)
  dataset[, mes_ant := shift(foto_mes, 1, type = "lag"), by = numero_de_cliente]
  dataset[, mes_sig := shift(foto_mes, 1, type = "lead"), by = numero_de_cliente]
  for (campo in campos_rotos) {
    dataset[, (campo) := as.numeric(get(campo))]
    dataset[, val_ant := shift(get(campo), 1, type = "lag"), by = numero_de_cliente]
    dataset[, val_sig := shift(get(campo), 1, type = "lead"), by = numero_de_cliente]
    dataset[foto_mes == 202006, (campo) := NA_real_]
    dataset[foto_mes == 202006 & mes_ant == 202005 & mes_sig == 202007,
      (campo) := (val_ant + val_sig) / 2]
  }
  dataset[, c("mes_ant", "mes_sig", "val_ant", "val_sig") := NULL]
  setorder(dataset, foto_mes, numero_de_cliente)

} else {
  stop("PARAM$CA$metodo desconocido")
}

# control: promedio de algunas variables rotas en 202005, 202006 y 202007
dataset[foto_mes %in% c(202005, 202006, 202007),
  lapply(.SD, function(x) mean(x, na.rm = TRUE)),
  by = foto_mes, .SDcols = head(campos_rotos, 4)]


#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202107  obviamente completo

* final_train =  [ 202005, 202105 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202105   completo, sin undersampling
   * training = [ 202005, 202104 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

La siguiente celda corre en interminables 8 minutos

In [ ]:
# preparo para que se pueda hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 60 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [ ]:
# Corrida 1: Grid Search completo (60 minutos)
# Corridas 2 y 3: se reutilizan los hiperparametros de la corrida 1 (ceteris paribus)
if (PARAM$gridsearch) {
  tb_nueva[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(.SD),
    by = 1:nrow(tb_nueva)]
} else {
  if (!file.exists(PARAM$archivo_hiper))
    stop("Primero hace la corrida 1 (asignar_NA) con PARAM$gridsearch <- TRUE")
  tb_nueva <- fread(PARAM$archivo_hiper)
}


la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

In [ ]:
# en la corrida 1 guardo los mejores hiperparametros para las otras corridas
if (PARAM$gridsearch) fwrite(tb_nueva[1], file = PARAM$archivo_hiper, sep = "\t")
PARAM$out$lgbm$mejores_hiperparametros


### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparámetros

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

#### Entrenamiento final y ganancia con 8 semillas
Cada semilla entrena un modelo distinto. La ganancia de cada una se agrega a `resultados_CA.txt`.

In [ ]:
# Entreno el modelo final con cada semilla y mido la ganancia en 202107
fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

PARAM$trainingstrategy$future <- c(202107)
dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

for (sem in PARAM$semillas) {
  param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)
  param_final$seed <- sem

  final_model <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)

  tb_prediccion <- dfuture[, list(numero_de_cliente, clase_ternaria)]
  tb_prediccion[, prob := predict(final_model, mfuture)]

  # ganancia en millones: +0.975 por BAJA+2, -0.025 por el resto
  tb_prediccion[, ganancia := -0.025]
  tb_prediccion[clase_ternaria == "BAJA+2", ganancia := 0.975]
  setorder(tb_prediccion, -prob)
  tb_prediccion[, gan_acum := cumsum(ganancia)]
  tb_prediccion[, gan_suavizada := frollmean(x = gan_acum, n = 400,
    align = "center", na.rm = TRUE, hasNA = TRUE)]

  linea <- data.table(
    fecha = format(Sys.time(), "%Y-%m-%d %H:%M"),
    metodo = PARAM$CA$metodo,
    semilla = sem,
    ganancia_suavizada_max = max(tb_prediccion$gan_suavizada, na.rm = TRUE),
    envios = which.max(tb_prediccion$gan_suavizada)
  )
  print(linea)
  flush.console()

  fwrite(linea, file = PARAM$archivo_resultados, sep = "\t", append = TRUE,
    col.names = !file.exists(PARAM$archivo_resultados))
  fwrite(tb_prediccion[, list(numero_de_cliente, prob, clase_ternaria, gan_acum)],
    file = paste0("ganancias_", sem, ".txt"), sep = "\t")

  rm(final_model)
  gc(full = TRUE, verbose = FALSE)
}


## Análisis final
Correr **solo cuando terminaste las 3 corridas**. Copiá la tabla y el gráfico a tus slides.

In [ ]:
# ===== ANALISIS: correr cuando terminaste las 3 corridas =====
require("data.table")
res <- fread("/content/buckets/b1/exp/resultados_CA.txt")

# 1. resumen por metodo
print(res[, list(semillas = .N,
  ganancia_media = round(mean(ganancia_suavizada_max), 2),
  desvio = round(sd(ganancia_suavizada_max), 2),
  envios_medio = round(mean(envios))), by = metodo])

# 2. tabla pareada (una fila por semilla)
ancho <- dcast(res, semilla ~ metodo, value.var = "ganancia_suavizada_max", fun.aggregate = mean)
print(ancho)

# 3. test de Wilcoxon pareado de cada metodo contra el de la catedra
for (m in setdiff(names(ancho), c("semilla", "asignar_NA"))) {
  ok <- complete.cases(ancho[, c("asignar_NA", m), with = FALSE])
  dif <- ancho[ok][[m]] - ancho[ok][["asignar_NA"]]
  wt <- wilcox.test(ancho[ok][[m]], ancho[ok][["asignar_NA"]], paired = TRUE)
  cat(m, "vs asignar_NA | semillas:", sum(ok),
    "| diferencia media:", round(mean(dif), 2),
    "| gana en", sum(dif > 0), "de", sum(ok), "semillas | p-value:", signif(wt$p.value, 3), "\n")
}

# 4. grafico para las slides
boxplot(ganancia_suavizada_max ~ metodo, data = res,
  main = "Ganancia en 202107 segun metodo de Catastrophe Analysis",
  ylab = "Ganancia suavizada maxima (millones)", xlab = "")
stripchart(ganancia_suavizada_max ~ metodo, data = res, vertical = TRUE,
  method = "jitter", add = TRUE, pch = 19, col = "steelblue")
